In [286]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2
import matplotlib.pyplot as plt

In [287]:
# Indlæs data
gdp = pd.read_csv("GDP.csv", skiprows=4)
aging = pd.read_csv("age-dependency-ratio-old.csv")
pwt = pd.read_excel("Human-Productivity.xlsx", sheet_name="Data")
rd = pd.read_csv("R&D.csv", skiprows=4)

# Giv ens kolonnenavne (meget vigtigt for merge senere)
gdp = gdp.rename(columns={"Country Name": "country", "Country Code": "countrycode"})
rd = rd.rename(columns={"Country Name": "country", "Country Code": "countrycode"})
aging = aging.rename(columns={"Country Name": "country", "Country Code": "countrycode"})

In [288]:
# Tjek de har læst korrekt - Kan sletts, kun til kontrol
print(gdp.head())
print(gdp.columns[:10])

print(aging.head())
print(aging.columns)

print(pwt.head())
print(pwt.columns[:15])

print(rd.head())
print(rd.columns[:15])

                       country countrycode  \
0                        Aruba         ABW   
1  Africa Eastern and Southern         AFE   
2                  Afghanistan         AFG   
3   Africa Western and Central         AFW   
4                       Angola         AGO   

                                  Indicator Name     Indicator Code  1960  \
0  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
1  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
2  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
3  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
4  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   

   1961  1962  1963  1964  1965  ...          2017          2018  \
0   NaN   NaN   NaN   NaN   NaN  ...  37524.914920  39287.059713   
1   NaN   NaN   NaN   NaN   NaN  ...   3837.726375   3723.216423   
2   NaN   NaN   NaN   NaN   NaN  ...   2335.795862   243

In [289]:
# Clean GDP data
gdp = gdp.loc[:, ~gdp.columns.str.contains("^Unnamed")]

# Behold kun relevante kolonner
gdp = gdp[["country", "countrycode"] + [col for col in gdp.columns if col.isdigit()]]

# Lav fra wide til long format
gdp = gdp.melt(
    id_vars=["country", "countrycode"],
    var_name="year",
    value_name="gdp_pc"
)

# Ens datatype
gdp["year"] = pd.to_numeric(gdp["year"], errors="coerce")
gdp["gdp_pc"] = pd.to_numeric(gdp["gdp_pc"], errors="coerce")

# Tjek (kan slettes senere)
print(gdp.head())
print(gdp.columns)
print(gdp.shape)
print(gdp.isna().sum())

                       country countrycode  year  gdp_pc
0                        Aruba         ABW  1960     NaN
1  Africa Eastern and Southern         AFE  1960     NaN
2                  Afghanistan         AFG  1960     NaN
3   Africa Western and Central         AFW  1960     NaN
4                       Angola         AGO  1960     NaN
Index(['country', 'countrycode', 'year', 'gdp_pc'], dtype='object')
(17556, 4)
country           0
countrycode       0
year              0
gdp_pc         9045
dtype: int64


In [290]:
# Clean aging
aging = aging.rename(columns={
    "Entity": "country",
    "Code": "countrycode",
    "Year": "year",
    "Age dependency ratio, old (% of working-age population)": "aging"
})

# Behold kun relevante kolonner
aging = aging[["country", "countrycode", "year", "aging"]]

# Ens datatype
aging["year"] = pd.to_numeric(aging["year"], errors="coerce")
aging["aging"] = pd.to_numeric(aging["aging"], errors="coerce")

# Fjern missing values i centrale variabler
aging = aging.dropna(subset=["countrycode", "aging"])

# Behold kun gyldige 3-bogstavs landekoder
aging = aging[aging["countrycode"].str.len() == 3]

# Tjek
print(aging.head())
print(aging.columns)
print(aging.shape)
print(aging.isna().sum())

       country countrycode  year     aging
0  Afghanistan         AFG  1950  5.078877
1  Afghanistan         AFG  1951  5.100585
2  Afghanistan         AFG  1952  5.114399
3  Afghanistan         AFG  1953  5.122446
4  Afghanistan         AFG  1954  5.126267
Index(['country', 'countrycode', 'year', 'aging'], dtype='object')
(17464, 4)
country        0
countrycode    0
year           0
aging          0
dtype: int64


In [291]:
# Clean PWT

# Behold relevante kolonner
pwt = pwt[["country", "countrycode", "year", "hc"]]

# Ens datatype
pwt["year"] = pd.to_numeric(pwt["year"], errors="coerce")
pwt["hc"] = pd.to_numeric(pwt["hc"], errors="coerce")

# Fjern missing values
pwt = pwt.dropna(subset=["hc"])

# Behold kun gyldige lande
pwt = pwt[pwt["countrycode"].str.len() == 3]

# Tjek (kan slettes senere)
print(pwt.head())
print(pwt.columns)
print(pwt.shape)
print(pwt.isna().sum())

   country countrycode  year        hc
94  Angola         AGO  1970  1.015686
95  Angola         AGO  1971  1.018196
96  Angola         AGO  1972  1.020712
97  Angola         AGO  1973  1.023234
98  Angola         AGO  1974  1.025762
Index(['country', 'countrycode', 'year', 'hc'], dtype='object')
(9217, 4)
country        0
countrycode    0
year           0
hc             0
dtype: int64


In [292]:
# Clean R&D
rd = rd.loc[:, ~rd.columns.str.contains("^Unnamed")]

# Behold kun relevante kolonner
rd = rd[["country", "countrycode"] + [col for col in rd.columns if col.isdigit()]]

# Lav fra wide til long format
rd = rd.melt(
    id_vars=["country", "countrycode"],
    var_name="year",
    value_name="rd_gdp"
)

# Ens datatype
rd["year"] = pd.to_numeric(rd["year"], errors="coerce")
rd["rd_gdp"] = pd.to_numeric(rd["rd_gdp"], errors="coerce")

# Fjern missing values
rd = rd.dropna(subset=["rd_gdp"])

# Behold kun gyldige lande
rd = rd[rd["countrycode"].str.len() == 3]

# Begræns periode
rd = rd[(rd["year"] >= 1990) & (rd["year"] <= 2020)]

# Tjek
print(rd.head())
print(rd.columns)
print(rd.shape)
print(rd.isna().sum())

         country countrycode  year   rd_gdp
9585   Argentina         ARG  1996  0.41749
9589   Australia         AUS  1996  1.66218
9590     Austria         AUT  1996  1.58947
9591  Azerbaijan         AZE  1996  0.23533
9593     Belgium         BEL  1996  1.74299
Index(['country', 'countrycode', 'year', 'rd_gdp'], dtype='object')
(2837, 4)
country        0
countrycode    0
year           0
rd_gdp         0
dtype: int64


In [293]:
# rd_sample = rd_long[["Country Code", "Year"]].drop_duplicates()

In [294]:
# Fjern evt. dubletter først
gdp = gdp.drop_duplicates(subset=["countrycode", "year"])
aging = aging.drop_duplicates(subset=["countrycode", "year"])
pwt = pwt.drop_duplicates(subset=["countrycode", "year"])
rd = rd.drop_duplicates(subset=["countrycode", "year"])

# Merge alle datasæt
df = aging.merge(gdp, on=["countrycode", "year"], how="inner")
df = df.merge(pwt, on=["countrycode", "year"], how="inner")
df = df.merge(rd[["countrycode", "year", "rd_gdp"]], on=["countrycode", "year"], how="inner")

# Tjek resultat
print(df.head())
print(df.columns)
print(df.shape)
print(df.isna().sum())

  country_x countrycode  year      aging country_y        gdp_pc  country  \
0   Albania         ALB  2007  14.793774   Albania   7584.689222  Albania   
1   Albania         ALB  2008  15.311145   Albania   8469.245375  Albania   
2   Algeria         DZA  2001   7.093249   Algeria   9543.520354  Algeria   
3   Algeria         DZA  2002   7.109899   Algeria  10080.182500  Algeria   
4   Algeria         DZA  2003   7.130257   Algeria  10802.750670  Algeria   

         hc   rd_gdp  
0  2.884521  0.08411  
1  2.891026  0.14973  
2  1.900015  0.21219  
3  1.915068  0.33807  
4  1.930241  0.18122  
Index(['country_x', 'countrycode', 'year', 'aging', 'country_y', 'gdp_pc',
       'country', 'hc', 'rd_gdp'],
      dtype='object')
(1943, 9)
country_x      0
countrycode    0
year           0
aging          0
country_y      0
gdp_pc         4
country        0
hc             0
rd_gdp         0
dtype: int64


In [295]:
# 1. Fjern dubletter
gdp   = gdp.drop_duplicates(subset=["countrycode", "year"])
aging = aging.drop_duplicates(subset=["countrycode", "year"])
pwt   = pwt.drop_duplicates(subset=["countrycode", "year"])
rd    = rd.drop_duplicates(subset=["countrycode", "year"])

# 2. Merge datasæt
df = aging.merge(gdp, on=["countrycode", "year"], how="inner")
df = df.merge(pwt, on=["countrycode", "year"], how="inner")
df = df.merge(rd[["countrycode", "year", "rd_gdp"]], on=["countrycode", "year"], how="inner")

# 3. Ryd op i country kolonner
df = df.drop(columns=["country_y", "country"], errors="ignore")
df = df.rename(columns={"country_x": "country"})

# 4. Fjern missing værdier (kun det nødvendige)
df = df.dropna(subset=["gdp_pc"])

# 5. Log af GDP
df["log_gdp_pc"] = np.log(df["gdp_pc"])

# 6. Endeligt tjek
print(df.head())
print(df.columns)
print(df.shape)
print(df.isna().sum())

   country countrycode  year      aging        gdp_pc        hc   rd_gdp  \
0  Albania         ALB  2007  14.793774   7584.689222  2.884521  0.08411   
1  Albania         ALB  2008  15.311145   8469.245375  2.891026  0.14973   
2  Algeria         DZA  2001   7.093249   9543.520354  1.900015  0.21219   
3  Algeria         DZA  2002   7.109899  10080.182500  1.915068  0.33807   
4  Algeria         DZA  2003   7.130257  10802.750670  1.930241  0.18122   

   log_gdp_pc  
0    8.933887  
1    9.044197  
2    9.163618  
3    9.218327  
4    9.287556  
Index(['country', 'countrycode', 'year', 'aging', 'gdp_pc', 'hc', 'rd_gdp',
       'log_gdp_pc'],
      dtype='object')
(1939, 8)
country        0
countrycode    0
year           0
aging          0
gdp_pc         0
hc             0
rd_gdp         0
log_gdp_pc     0
dtype: int64


In [296]:
df = aging.merge(gdp, on=["countrycode", "year"], how="inner")
df = df.merge(pwt, on=["countrycode", "year"], how="inner")
df = df.merge(rd[["countrycode", "year", "rd_gdp"]], on=["countrycode", "year"], how="inner")

In [297]:
# Ryd op i country-kolonner efter merge
df = df.drop(columns=["country_y", "country"], errors="ignore")
df = df.rename(columns={"country_x": "country"})

# Fjern evt. manglende BNP-værdier
df = df.dropna(subset=["gdp_pc"])

# Lav log af BNP per capita
df["log_gdp_pc"] = np.log(df["gdp_pc"])

# Tjek det merged datasæt
print(df.head())
print(df.columns)
print(df.shape)
print(df.isna().sum())

   country countrycode  year      aging        gdp_pc        hc   rd_gdp  \
0  Albania         ALB  2007  14.793774   7584.689222  2.884521  0.08411   
1  Albania         ALB  2008  15.311145   8469.245375  2.891026  0.14973   
2  Algeria         DZA  2001   7.093249   9543.520354  1.900015  0.21219   
3  Algeria         DZA  2002   7.109899  10080.182500  1.915068  0.33807   
4  Algeria         DZA  2003   7.130257  10802.750670  1.930241  0.18122   

   log_gdp_pc  
0    8.933887  
1    9.044197  
2    9.163618  
3    9.218327  
4    9.287556  
Index(['country', 'countrycode', 'year', 'aging', 'gdp_pc', 'hc', 'rd_gdp',
       'log_gdp_pc'],
      dtype='object')
(1939, 8)
country        0
countrycode    0
year           0
aging          0
gdp_pc         0
hc             0
rd_gdp         0
log_gdp_pc     0
dtype: int64


In [298]:
# Print kolonnerne i df
print(df.columns)

Index(['country', 'countrycode', 'year', 'aging', 'gdp_pc', 'hc', 'rd_gdp',
       'log_gdp_pc'],
      dtype='object')


##  Deskriptiv statistik 

In [299]:
# Deskriptive statistikker
print(df.describe())

              year        aging         gdp_pc           hc       rd_gdp  \
count  1939.000000  1939.000000    1939.000000  1939.000000  1939.000000   
mean   2008.642084    16.413474   24089.892467     2.823995     1.006133   
std       6.940838     8.756887   21205.669553     0.583118     0.974204   
min    1996.000000     1.219402     455.866557     1.053331     0.005440   
25%    2003.000000     8.355403    8804.162501     2.476312     0.248365   
50%    2009.000000    16.261354   19071.444420     2.927319     0.660200   
75%    2015.000000    23.574267   33415.651368     3.249656     1.536455   
max    2020.000000    48.979670  180939.439450     3.854527     5.832090   

        log_gdp_pc  
count  1939.000000  
mean      9.657903  
std       1.049098  
min       6.122200  
25%       9.082980  
50%       9.855947  
75%      10.416780  
max      12.105918  


In [300]:
# Sortér data
#df = df.sort_values(["countrycode", "year"])

# Beregn årlig vækst i GDP per capita
#df["gdp_growth"] = df.groupby("countrycode")["gdp_pc"].pct_change()

# Identificer recessioner
#df["recession"] = (df["gdp_growth"] < 0).astype(int)

# Drop rækker med manglende værdier i gdp_growth
#df = df.dropna(subset=["gdp_growth"])

# Tjek
#print(df[["country", "countrycode", "year", "gdp_pc", "gdp_growth", "recession"]].head())
#print(df[["gdp_growth", "recession"]].describe())

In [302]:
# kan slettes, kun til kontrol
df[["country", "countrycode", "year", "gdp_pc", "hc", "rd_gdp"]].head(20)

,country,countrycode,year,gdp_pc,hc,rd_gdp
0,Albania,ALB,2007,7584.689222,2.884521,0.08411
1,Albania,ALB,2008,8469.245375,2.891026,0.14973
2,Algeria,DZA,2001,9543.520354,1.900015,0.21219
3,Algeria,DZA,2002,10080.182500,1.915068,0.33807
4,Algeria,DZA,2003,10802.750670,1.930241,0.18122
5,Algeria,DZA,2004,11431.685672,1.945534,0.15186
6,Algeria,DZA,2005,12246.269224,1.960948,0.06367
7,Algeria,DZA,2017,13493.560749,2.302799,0.47865
8,Angola,AGO,2016,7766.837875,1.460044,0.03229
9,Argentina,ARG,1996,10495.839245,2.627024,0.41749


In [303]:
print(pwt.columns)

Index(['country', 'countrycode', 'year', 'hc'], dtype='object')


### OLS regression (Ordinary Least Squares)

In [305]:
# Aldring → teknologi
model1 = smf.ols("rd_gdp ~ aging + hc", data=df).fit(cov_type="HC1")
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:                 rd_gdp   R-squared:                       0.417
Model:                            OLS   Adj. R-squared:                  0.417
Method:                 Least Squares   F-statistic:                     810.1
Date:                Tue, 14 Apr 2026   Prob (F-statistic):          2.38e-256
Time:                        11:45:30   Log-Likelihood:                -2176.3
No. Observations:                1939   AIC:                             4359.
Df Residuals:                    1936   BIC:                             4375.
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.3200      0.089    -14.874      0.0

In [ ]:
# Teknologi + R&D 
model2 = smf.ols("log_gdp_pc ~ rd_gdp + aging + hc", data=df).fit(cov_type="HC1")
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:             log_gdp_pc   R-squared:                       0.485
Model:                            OLS   Adj. R-squared:                  0.484
Method:                 Least Squares   F-statistic:                     642.5
Date:                Tue, 14 Apr 2026   Prob (F-statistic):          8.95e-290
Time:                        11:46:25   Log-Likelihood:                -2200.6
No. Observations:                1939   AIC:                             4409.
Df Residuals:                    1935   BIC:                             4431.
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      6.8410      0.111     61.695      0.0

### FE

In [307]:
from linearmodels.panel import PanelOLS

df_panel = df.set_index(["countrycode", "year"])

fe1 = PanelOLS.from_formula(
    "rd_gdp ~ aging + hc + EntityEffects + TimeEffects",
    data=df_panel
).fit(cov_type="robust")

print(fe1.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 rd_gdp   R-squared:                        0.0636
Estimator:                   PanelOLS   R-squared (Between):              0.6017
No. Observations:                1939   R-squared (Within):               0.2702
Date:                Tue, Apr 14 2026   R-squared (Overall):              0.6531
Time:                        11:51:35   Log-likelihood                    277.17
Cov. Estimator:                Robust                                           
                                        F-statistic:                      60.712
Entities:                         126   P-value                           0.0000
Avg Obs:                       15.389   Distribution:                  F(2,1787)
Min Obs:                       1.0000                                           
Max Obs:                       25.000   F-statistic (robust):             39.260
                            

In [308]:
model2_fe = PanelOLS.from_formula(
    "log_gdp_pc ~ rd_gdp + aging + hc + EntityEffects + TimeEffects",
    data=df_panel
).fit(cov_type="robust")

print(model2_fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:             log_gdp_pc   R-squared:                        0.0085
Estimator:                   PanelOLS   R-squared (Between):             -0.0254
No. Observations:                1939   R-squared (Within):              -0.0756
Date:                Tue, Apr 14 2026   R-squared (Overall):             -0.0304
Time:                        11:52:53   Log-likelihood                    986.79
Cov. Estimator:                Robust                                           
                                        F-statistic:                      5.1146
Entities:                         126   P-value                           0.0016
Avg Obs:                       15.389   Distribution:                  F(3,1786)
Min Obs:                       1.0000                                           
Max Obs:                       25.000   F-statistic (robust):             3.8297
                            

In [309]:
df_panel["rd_lag"] = df_panel.groupby(level=0)["rd_gdp"].shift(1)

model_lag = PanelOLS.from_formula(
    "log_gdp_pc ~ rd_lag + aging + hc + EntityEffects + TimeEffects",
    data=df_panel
).fit(cov_type="robust")

print(model_lag.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:             log_gdp_pc   R-squared:                        0.0085
Estimator:                   PanelOLS   R-squared (Between):             -0.0311
No. Observations:                1813   R-squared (Within):              -0.0865
Date:                Tue, Apr 14 2026   R-squared (Overall):             -0.0361
Time:                        11:56:02   Log-likelihood                    963.08
Cov. Estimator:                Robust                                           
                                        F-statistic:                      4.7937
Entities:                         119   P-value                           0.0025
Avg Obs:                       15.235   Distribution:                  F(3,1668)
Min Obs:                       1.0000                                           
Max Obs:                       24.000   F-statistic (robust):             3.6614
                            

/opt/anaconda3/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


### RE

In [310]:
from linearmodels.panel import RandomEffects

re = RandomEffects.from_formula(
    "log_gdp_pc ~ rd_gdp + aging + hc",
    data=df_panel
).fit()

print(re.summary)


                        RandomEffects Estimation Summary                        
Dep. Variable:             log_gdp_pc   R-squared:                        0.7417
Estimator:              RandomEffects   R-squared (Between):              0.8709
No. Observations:                1939   R-squared (Within):               0.6317
Date:                Tue, Apr 14 2026   R-squared (Overall):              0.8978
Time:                        12:02:43   Log-likelihood                   -23.836
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      1853.3
Entities:                         126   P-value                           0.0000
Avg Obs:                       15.389   Distribution:                  F(3,1936)
Min Obs:                       1.0000                                           
Max Obs:                       25.000   F-statistic (robust):             1853.3
                            

### Hausman test

In [311]:
import numpy as np
from scipy.stats import chi2

# Vælg fælles variable
vars = ["rd_gdp", "aging", "hc"]

# Hent koefficienter
fe_params = model2_fe.params[vars]
re_params = re.params[vars]

# Hent kovariansmatricer
fe_cov = model2_fe.cov.loc[vars, vars]
re_cov = re.cov.loc[vars, vars]

# Beregn forskel
diff = fe_params - re_params

# Hausman test statistik
stat = np.dot(np.dot(diff.T, np.linalg.inv(fe_cov - re_cov)), diff)

# Frihedsgrader
df_h = len(diff)

# p-value
p_value = 1 - chi2.cdf(stat, df_h)

print("Hausman test statistic:", stat)
print("p-value:", p_value)

Hausman test statistic: 866.1406193922709
p-value: 0.0
